Migrated from Databricks to Neuron (auto-port).

In [ ]:
import os
import sys
from pathlib import Path

for _candidate in (
    "/home/jovyan/work/default/rashmi-ranjan/lib",
    str(Path.cwd().parent / "lib"),
    str(Path.cwd() / "lib"),
):
    if _candidate not in sys.path and Path(_candidate).is_dir():
        sys.path.insert(0, _candidate)

from neuron_runtime import (
    GoogleSheetReader,
    SlackReportsTrigger,
    get_sql_df,
    load_service_account,
    load_slack_token,
    make_unique_columns,
)

service_account = load_service_account()
slack_token = load_slack_token()

In [ ]:
import pandas as pd

query = """
WITH

base AS (
    SELECT
        m.sub_ticket_id,
        m.ticket_id,
        m.canonical_ticket_id,
        m.customer_id,
        m.ticket_group,
        m.ticket_state,
        m.source_channel,
        m.overall_last_agent_type_for_sub_ticket,
        m.last_obcs_l1_agent_id,
        m.last_obcs_l1_agent_name,
        m.overall_last_agent_id_for_sub_ticket,
        m.overall_last_agent_name_for_sub_ticket,
        m.ticket_created_dt,
        m.phone_number,
        m.ticket_created_time,
        m.resolved_time,
        m.last_agent_id_for_reporting,
        m.last_agent_name_for_reporting,
        o.origination_reason,
        o.origination_sub_reason
    FROM (
        SELECT
            sub_ticket_id,
            ticket_id,
            canonical_ticket_id,
            customer_id,
            ticket_group,
            ticket_state,
            source_channel,
            overall_last_agent_type_for_sub_ticket,
            last_obcs_l1_agent_id,
            last_obcs_l1_agent_name,
            overall_last_agent_id_for_sub_ticket,
            overall_last_agent_name_for_sub_ticket,
            ticket_created_dt,
            phone_number,
            ticket_created_time,
            resolved_time,
            CASE
                WHEN overall_last_agent_type_for_sub_ticket = 'CLAIMS_CALL_SUPPORT_AGENT_L1'
                    THEN coalesce(overall_last_agent_id_for_sub_ticket, last_obcs_l1_agent_id)
                ELSE last_obcs_l1_agent_id
            END AS last_agent_id_for_reporting,
            CASE
                WHEN overall_last_agent_type_for_sub_ticket = 'CLAIMS_CALL_SUPPORT_AGENT_L1'
                    THEN coalesce(overall_last_agent_name_for_sub_ticket, last_obcs_l1_agent_name)
                ELSE last_obcs_l1_agent_name
            END AS last_agent_name_for_reporting
        FROM corp_analytics.cx_sub_ticket_data_model_tbl
        WHERE ticket_created_dt >= date_format(current_date - interval '30' day, '%Y%m%d')
          AND overall_last_agent_type_for_sub_ticket = 'CLAIMS_CALL_SUPPORT_AGENT_L1'
    ) m
    LEFT JOIN (
        SELECT
            tk.canonical_ticket_id,
            arbitrary(tk.origination_reason) AS origination_reason,
            arbitrary(tk.origination_sub_reason) AS origination_sub_reason
        FROM crm_timekeeper.ticket_event_audit t
        INNER JOIN crm_tickets.tickets tk
            ON t.ticket_id = tk.id
        WHERE tk.agent_type = 'CLAIMS_CALL_SUPPORT_AGENT_L1'
          AND t.dispositions IS NOT NULL
        GROUP BY tk.canonical_ticket_id
    ) o
        ON CAST(m.canonical_ticket_id AS varchar) = CAST(o.canonical_ticket_id AS varchar)
),

crm_l1_excluded AS (
    SELECT DISTINCT
        CAST(tk.id AS varchar) AS crm_ticket_id,
        CAST(tk.canonical_ticket_id AS varchar) AS canonical_ticket_id
    FROM crm_tickets.tickets tk
    LEFT JOIN crm_timekeeper.ticket_event_audit t
        ON t.ticket_id = tk.id
    WHERE tk.agent_type = 'CLAIMS_CALL_SUPPORT_AGENT_L1'
      AND (
            lower(trim(CAST(tk.origination_reason AS varchar))) IN (
                'not connected',
                'non claim related'
            )
         OR lower(trim(CAST(tk.origination_sub_reason AS varchar))) IN (
                'short call',
                'others',
                'other',
                'non insurance related',
                'loan related',
                'upi related',
                'test call',
                'rnr / busy',
                'rnr/ busy',
                'rnr/switched off/ busy/ not reachalbe',
                'rnr/switched off/ busy/ not reachable',
                'rnr/switched off',
                'upi /trezo / bbps',
                'cash loan / pl',
                'home loan',
                'not connected /rnr/swithch off /busy'
            )
         OR lower(coalesce(CAST(t.dispositions AS varchar), '')) LIKE '%not connected%'
         OR lower(coalesce(CAST(t.dispositions AS varchar), '')) LIKE '%non claim related%'
         OR lower(coalesce(CAST(t.dispositions AS varchar), '')) LIKE '%short call%'
         OR lower(coalesce(CAST(t.dispositions AS varchar), '')) LIKE '%others%'
         OR lower(coalesce(CAST(t.dispositions AS varchar), '')) LIKE '%non insurance related%'
         OR lower(coalesce(CAST(t.dispositions AS varchar), '')) LIKE '%loan related%'
         OR lower(coalesce(CAST(t.dispositions AS varchar), '')) LIKE '%upi related%'
         OR lower(coalesce(CAST(t.dispositions AS varchar), '')) LIKE '%test call%'
         OR lower(coalesce(CAST(t.dispositions AS varchar), '')) LIKE '%rnr/ busy%'
         OR lower(coalesce(CAST(t.dispositions AS varchar), '')) LIKE '%rnr / busy%'
         OR lower(coalesce(CAST(t.dispositions AS varchar), '')) LIKE '%rnr/switched off%'
         OR lower(coalesce(CAST(t.dispositions AS varchar), '')) LIKE '%upi /trezo / bbps%'
         OR lower(coalesce(CAST(t.dispositions AS varchar), '')) LIKE '%cash loan / pl%'
         OR lower(coalesce(CAST(t.dispositions AS varchar), '')) LIKE '%home loan%'
         OR lower(coalesce(CAST(t.dispositions AS varchar), '')) LIKE '%not connected /rnr/swithch off /busy%'
      )
),

-- GI Outbound Connected Calls (Query 2)
gi_outbound_tickets AS (
    SELECT
        t.id AS ticket_id,
        json_extract_scalar(t.meta_data, '$.call_id') AS call_id,
        json_extract_scalar(t.meta_data, '$.call_flow') AS call_flow,
        date(
            from_iso8601_timestamp(CAST(t.created_at AS varchar)) AT TIME ZONE 'Asia/Kolkata'
        ) AS call_date_ist,
        from_iso8601_timestamp(CAST(t.created_at AS varchar)) AT TIME ZONE 'Asia/Kolkata' AS call_timestamp_ist,
        t.ticket_state,
        CASE
            WHEN json_extract_scalar(t.meta_data, '$.call_flow') = 'INBOUND' THEN 'INBOUND'
            ELSE 'OUTBOUND'
        END AS call_direction
    FROM crm_tickets.tickets t
    WHERE t.ticket_group = 'GENERAL_INSURANCE'
      AND t.agent_type = 'CLAIMS_CALL_SUPPORT_AGENT_L1'
      AND t.source_channel = 'CALL'
),

gi_outbound_agent_lookup AS (
    SELECT
        m.ticket_id,
        max(m.overall_last_agent_id_for_ticket) AS agent_id,
        max(m.overall_last_agent_type_for_ticket) AS overall_last_agent_type_for_ticket,
        max(m.canonical_ticket_id) AS canonical_ticket_id
    FROM corp_analytics.cx_master_data_model_tbl m
    WHERE m.partition_key >= date_format(current_date - interval '120' day, '%Y%m%d')
    GROUP BY m.ticket_id
),

gi_outbound_calls AS (
    SELECT
        t.ticket_id,
        al.canonical_ticket_id,
        t.call_date_ist,
        t.call_timestamp_ist,
        u.name AS agent_name,
        u.email_id AS agent_email,
        date(
            at_timezone(
                coalesce(
                    try(from_iso8601_timestamp(CAST(cl.connected_at_timestamp AS varchar))),
                    cast(cl.connected_at_timestamp AS timestamp)
                ),
                'Asia/Kolkata'
            )
        ) AS resolved_date,
        at_timezone(
            coalesce(
                try(from_iso8601_timestamp(CAST(cl.connected_at_timestamp AS varchar))),
                cast(cl.connected_at_timestamp AS timestamp)
            ),
            'Asia/Kolkata'
        ) AS resolved_timestamp_ist
    FROM gi_outbound_tickets t
    LEFT JOIN gi_outbound_agent_lookup al
        ON al.ticket_id = t.ticket_id
    LEFT JOIN mjolnir_service.users u
        ON u.account_id = al.agent_id
       AND u.__deleted = 'false'
    INNER JOIN graham_bell_service.call_logs cl
        ON cl.call_id = t.call_id
    WHERE t.call_direction = 'OUTBOUND'
      AND cl.connected_at_timestamp IS NOT NULL
      AND date(
            at_timezone(
                coalesce(
                    try(from_iso8601_timestamp(CAST(cl.connected_at_timestamp AS varchar))),
                    cast(cl.connected_at_timestamp AS timestamp)
                ),
                'Asia/Kolkata'
            )
        ) >= current_date - interval '90' day
),

query1 AS (
    SELECT
        date_add(
            'day',
            CASE
                WHEN day_of_week(CAST(resolved_date AS date)) = 6 THEN 2
                WHEN day_of_week(CAST(resolved_date AS date)) = 7 THEN 1
                ELSE 1
            END,
            CAST(resolved_date AS date)
        ) AS Audit_Date,
        CAST(resolved_date AS date) AS resolved_date,
        group_name,
        COUNT(*) AS count_of_interactions
    FROM (
        SELECT resolved_date, ticket_id, group_name
        FROM (
            SELECT
                s.resolved_date,
                s.ticket_id,
                s.group_name,
                row_number() OVER (
                    PARTITION BY s.ticket_id
                    ORDER BY s.resolved_date DESC NULLS LAST, s.group_name
                ) AS rn
            FROM (

                SELECT
                    date(
                        coalesce(
                            try(from_iso8601_timestamp(nullif(trim(CAST(hit.resolved_date AS varchar)), ''))),
                            try(from_iso8601_timestamp(nullif(trim(CAST(hit.closed_date AS varchar)), '')))
                        )
                    ) AS resolved_date,
                    hit.ticket_id,
                    CASE
                        WHEN hit.group_name = 'Claims Support'
                             AND fd.call_email_disposition = 'Rejection Rationale'
                        THEN 'Rejection Calling'
                        ELSE hit.group_name
                    END AS group_name
                FROM omnitrix.hi_tickets_v1 hit
                LEFT JOIN omnitrix.freshdesk_ticket_gi_report fd
                    ON fd.ticket_id = hit.ticket_id
                WHERE date(
                        coalesce(
                            try(from_iso8601_timestamp(nullif(trim(CAST(hit.resolved_date AS varchar)), ''))),
                            try(from_iso8601_timestamp(nullif(trim(CAST(hit.closed_date AS varchar)), '')))
                        )
                    ) >= current_date - interval '90' day
                  AND (
                        (
                            hit.status IN ('Resolved', 'Closed')
                            AND hit.group_name IN (
                                'Senior Callback', 'Claims Support',
                                'Rejection Calling', 'Claims De-escalation'
                            )
                        )
                        OR (
                            hit.status = 'Resolved'
                            AND hit.group_name = 'Claims Support Emails'
                        )
                  )
                  AND (
                        (
                            hit.status = 'Resolved'
                            AND hit.group_name = 'Claims Support Emails'
                        )
                        OR (
                            (
                                hit.group_name NOT IN ('Rejection Calling', 'Senior Callback')
                                AND fd.claims_type NOT IN ('Not Connected')
                            )
                            OR (
                                hit.group_name IN ('Rejection Calling', 'Senior Callback')
                                AND fd.call_status = 'Connected'
                            )
                        )
                  )
                  AND coalesce(trim(CAST(hit.agent_name AS varchar)), '') <> 'No Agent'
                  AND NOT EXISTS (
                        SELECT 1
                        FROM crm_l1_excluded e
                        WHERE CAST(hit.ticket_id AS varchar) = e.crm_ticket_id
                           OR CAST(hit.ticket_id AS varchar) = e.canonical_ticket_id
                  )

                UNION ALL

                SELECT
                    date(
                        try(from_iso8601_timestamp(nullif(trim(CAST(hit.resolved_date AS varchar)), '')))
                    ) AS resolved_date,
                    hit.ticket_id,
                    hit.group_name
                FROM omnitrix.hi_tickets_v1 hit
                LEFT JOIN omnitrix.freshdesk_ticket_gi_report fd
                    ON fd.ticket_id = hit.ticket_id
                WHERE hit.group_name IN ('BIMA Bharosa Portal', 'manager.experience', 'Grievance Team')
                  AND date(
                        try(from_iso8601_timestamp(nullif(trim(CAST(hit.resolved_date AS varchar)), '')))
                    ) >= current_date - interval '90' day
                  AND hit.status = 'Resolved'
                  AND coalesce(trim(CAST(hit.agent_name AS varchar)), '') <> 'No Agent'
                  AND NOT EXISTS (
                      SELECT 1
                      FROM crm_l1_excluded e
                      WHERE CAST(hit.ticket_id AS varchar) = e.crm_ticket_id
                         OR CAST(hit.ticket_id AS varchar) = e.canonical_ticket_id
                  )

                UNION ALL

                SELECT
                    date(
                        coalesce(
                            try(from_iso8601_timestamp(nullif(trim(CAST(hit.resolved_date AS varchar)), ''))),
                            try(from_iso8601_timestamp(nullif(trim(CAST(hit.closed_date AS varchar)), '')))
                        )
                    ) AS resolved_date,
                    hit.ticket_id,
                    CASE
                        WHEN hit.group_name = 'Claims Support'
                             AND fd.call_email_disposition = 'Rejection Rationale'
                        THEN 'Rejection Calling'
                        ELSE 'Motor Claims'
                    END AS group_name
                FROM omnitrix.hi_tickets_v1 hit
                LEFT JOIN omnitrix.freshdesk_ticket_gi_report fd
                    ON fd.ticket_id = hit.ticket_id
                WHERE (
                        (hit.group_name = 'Claims Support' AND fd.claims_lob = 'Motor Claims')
                        OR hit.group_name IN ('Motor Claims - Emails', 'Motor Claims - Outbound Call')
                    )
                  AND date(
                        coalesce(
                            try(from_iso8601_timestamp(nullif(trim(CAST(hit.resolved_date AS varchar)), ''))),
                            try(from_iso8601_timestamp(nullif(trim(CAST(hit.closed_date AS varchar)), '')))
                        )
                    ) >= current_date - interval '90' day
                  AND hit.status IN ('Resolved', 'Closed')
                  AND hit.group_name IN ('Claims Support', 'Motor Claims - Outbound Call', 'Motor Claims - Emails')
                  AND (
                        hit.group_name NOT IN ('Motor Claims - Emails', 'Motor Claims - Outbound Call')
                        OR coalesce(trim(CAST(hit.agent_email AS varchar)), '') <> ''
                    )
                  AND coalesce(trim(CAST(hit.agent_name AS varchar)), '') <> 'No Agent'
                  AND NOT EXISTS (
                      SELECT 1
                      FROM crm_l1_excluded e
                      WHERE CAST(hit.ticket_id AS varchar) = e.crm_ticket_id
                         OR CAST(hit.ticket_id AS varchar) = e.canonical_ticket_id
                  )

                UNION ALL

                SELECT
                    date(
                        coalesce(
                            try(from_iso8601_timestamp(CAST(b.resolved_time AS varchar))),
                            try(cast(b.resolved_time AS timestamp))
                        )
                    ) AS resolved_date,
                    CAST(b.canonical_ticket_id AS varchar) AS ticket_id,
                    'Claims Support' AS group_name
                FROM base b
                WHERE b.resolved_time IS NOT NULL
                  AND date(
                        coalesce(
                            try(from_iso8601_timestamp(CAST(b.resolved_time AS varchar))),
                            try(cast(b.resolved_time AS timestamp))
                        )
                    ) >= current_date - interval '90' day
                  AND (
                        b.origination_reason IS NULL
                        OR lower(trim(CAST(b.origination_reason AS varchar))) NOT IN (
                            'not connected',
                            'non claim related'
                        )
                  )
                  AND (
                        b.origination_sub_reason IS NULL
                        OR lower(trim(CAST(b.origination_sub_reason AS varchar))) NOT IN (
                            'short call',
                            'others',
                            'other',
                            'non insurance related',
                            'loan related',
                            'upi related',
                            'test call',
                            'rnr / busy',
                            'rnr/ busy',
                            'rnr/switched off/ busy/ not reachalbe',
                            'rnr/switched off/ busy/ not reachable',
                            'rnr/switched off',
                            'upi /trezo / bbps',
                            'cash loan / pl',
                            'home loan',
                            'not connected /rnr/swithch off /busy'
                        )
                  )
                  AND coalesce(trim(CAST(b.last_agent_name_for_reporting AS varchar)), '') <> 'No Agent'
                  AND NOT EXISTS (
                      SELECT 1
                      FROM crm_l1_excluded e
                      WHERE CAST(b.canonical_ticket_id AS varchar) = e.canonical_ticket_id
                         OR CAST(b.ticket_id AS varchar) = e.crm_ticket_id
                         OR CAST(b.ticket_id AS varchar) = e.canonical_ticket_id
                         OR CAST(b.sub_ticket_id AS varchar) = e.crm_ticket_id
                         OR CAST(b.sub_ticket_id AS varchar) = e.canonical_ticket_id
                  )

                UNION ALL

                -- GI Outbound Connected Calls → counted under Claims Support (additive)
                SELECT
                    goc.resolved_date AS resolved_date,
                    CONCAT(CAST(goc.canonical_ticket_id AS varchar), '_GI_OUTBOUND') AS ticket_id,
                    'Claims Support' AS group_name
                FROM gi_outbound_calls goc
                WHERE goc.resolved_date >= current_date - interval '90' day
                  AND goc.canonical_ticket_id IS NOT NULL
                  AND coalesce(trim(CAST(goc.agent_name AS varchar)), '') <> 'No Agent'
                  AND NOT EXISTS (
                      SELECT 1
                      FROM crm_l1_excluded e
                      WHERE CAST(goc.canonical_ticket_id AS varchar) = e.canonical_ticket_id
                         OR CAST(goc.ticket_id AS varchar) = e.crm_ticket_id
                         OR CAST(goc.ticket_id AS varchar) = e.canonical_ticket_id
                  )

            ) s
        ) ranked
        WHERE rn = 1
          AND resolved_date < current_date
    ) deduped_per_ticket
    GROUP BY 1, 2, 3
),

query2 AS (
    SELECT
        date_add(
            'day',
            CASE
                WHEN day_of_week(CAST(dt AS date)) = 6 THEN 2
                WHEN day_of_week(CAST(dt AS date)) = 7 THEN 1
                ELSE 1
            END,
            CAST(dt AS date)
        ) AS Audit_Date,
        CAST(dt AS date) AS resolved_date,
        'ME Audits' AS group_name,
        COUNT(id) AS count_of_interactions
    FROM (
        SELECT
            id, dt, me_name,
            row_number() OVER (
                PARTITION BY id, audit_time, from_status, to_status, me_name
                ORDER BY claim_last_date_trail_new DESC
            ) AS rk
        FROM (
            SELECT
                x.id AS id,
                x.claim_last_date_trail_new,
                c.done_by AS me_name,
                c.from_status,
                c.to_status,
                from_iso8601_timestamp(CAST(c.audit_time AS varchar)) AS audit_time,
                date(from_iso8601_timestamp(CAST(c.audit_time AS varchar))) AS dt
            FROM gi_claims.claim_audit_trail c
            JOIN corp_analytics.ss_gi_claims_all_tbl x
                ON c.claim_id = x.id
            WHERE (
                    (c.from_status = 'PENDING_DATA_ENTRY_ME' AND c.to_status = 'PENDING_ADJUDICATION')
                 OR (c.from_status = 'PENDING_REJECTION_LETTER_CREATION' AND c.to_status = 'PENDING_REJECTION_LETTER_REVIEW')
                )
              AND x.year_week_crd NOT IN ('Null', 'null')
        ) base_table
    ) base_date
    WHERE rk = 1
      AND dt >= current_date - interval '90' day
      AND dt < current_date
      AND me_name IS NOT NULL
    GROUP BY 1, 2, 3
)

SELECT *
FROM (
    SELECT * FROM query1
    UNION ALL
    SELECT * FROM query2
) final
ORDER BY Audit_Date DESC, resolved_date DESC, group_name
"""

result = get_sql_df(query)

if isinstance(result, tuple):
    df, code = result
else:
    df = result
    code = None

if df is not None and not df.empty:
    display(df)
else:
    print("No rows returned.", f"Code={code}" if code is not None else "")

In [ ]:
# Python cell
import os
import shutil

if not os.path.exists("data"):
    os.makedirs("data")

gsheet_url = "https://docs.google.com/spreadsheets/d/1DwWg3qdCRLI9k4rmDeMfH1PGv8XPoXsxTjZ88gvQpjA/edit?pli=1&gid=1994166085#gid=1994166085"
worksheet_name = "Connected Call Count (Except SC)"
gs_reader = GoogleSheetReader(gsheet_url, service_account)

gs_reader.clear_sheets(
    worksheet_name,
    start_row=1,
    start_column=1
)

combined_df = pd.concat([df], ignore_index=True) 

gs_reader.set_df_in_sheets( 
worksheet_name, 
combined_df, 
include_headers=True, 
row=1, 
col=1
)